# Simple XGBoost model: Biological Age
5 CSV files (demographics, laboratory, examination, dietary, questionnaire) ni `ml_model/data/` folder lo petandi (Colab lo aithe Files lo upload cheyyandi).
Idi website form lo unna attributes tho matrame train chestundi, so website inputs and model match avuthayi.
Run ayyaka `biological_age_xgboost_clean.pkl` ml_model folder lo save avuthundi.

In [1]:
%pip install xgboost scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os, pickle, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

DATA = "data/" if os.path.isdir("data") else "./"
assert os.path.exists(DATA + "demographics.csv"), "5 CSV files ni ml_model/data folder lo petandi"

FEATURES = ["RIAGENDR","RIDRETH1","BMXHT","BMXWT","BMXWAIST","BMXBMI","BPXSY1","BPXDI1","BPXPLS",
            "LBXGH","LBXSGL","LBXSCR","LBXTC","LBDHDD","LBXSAL","LBXHGB","LBXWBCSI","LBXPLTSI",
            "LBXSATSI","LBXSASSI","LBXSUA","LBXSBU","BPQ020","BPQ080","DIQ010","SMQ020","RHQ031"]

df = pd.read_csv(DATA + "demographics.csv", low_memory=False)[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH1"]]
for f in ["laboratory", "examination", "dietary", "questionnaire"]:
    head = pd.read_csv(DATA + f + ".csv", nrows=0).columns
    cols = ["SEQN"] + [c for c in FEATURES if c in head and c not in df.columns]
    df = df.merge(pd.read_csv(DATA + f + ".csv", usecols=cols, low_memory=False), on="SEQN", how="left")
print(df.shape, "| missing columns:", [c for c in FEATURES if c not in df.columns])

(9254, 29) | missing columns: []


In [3]:
df = df[(df.RIDAGEYR >= 18) & df.RIDAGEYR.notna()]
FEATURES = [c for c in FEATURES if c in df.columns]
X = df[FEATURES].astype(float)
y = df.RIDAGEYR.values
print("rows:", len(X), "features:", len(FEATURES))

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
Xfit, Xval, yfit, yval = train_test_split(Xtr, ytr, test_size=0.15, random_state=7)

rows: 5856 features: 27


In [4]:
model = XGBRegressor(n_estimators=2000, learning_rate=0.03, max_depth=4, subsample=0.8,
                     colsample_bytree=0.8, min_child_weight=3, reg_lambda=3,
                     tree_method="hist", random_state=0, n_jobs=-1, early_stopping_rounds=50)
model.fit(Xfit, yfit, eval_set=[(Xval, yval)], verbose=False)

p = model.predict(Xte)
print(f"TEST  R2={r2_score(yte, p):.3f}  MAE={mean_absolute_error(yte, p):.2f} yrs  "
      f"RMSE={mean_squared_error(yte, p) ** .5:.2f}")

TEST  R2=0.687  MAE=8.16 yrs  RMSE=10.55


In [5]:
res = pd.DataFrame({"real_age": yte, "biological_age": p.round(1)})
res["age_gap"] = (res.biological_age - res.real_age).round(1)
print(res.head(8))
print(pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False).head(10))

   real_age  biological_age  age_gap
0      73.0       57.099998    -15.9
1      18.0       23.799999      5.8
2      39.0       37.099998     -1.9
3      26.0       41.099998     15.1
4      31.0       34.799999      3.8
5      19.0       33.299999     14.3
6      63.0       54.299999     -8.7
7      56.0       61.000000      5.0
RHQ031      0.235187
BPQ020      0.213004
BPQ080      0.106361
LBXGH       0.075752
BPXSY1      0.046399
LBXSBU      0.045650
DIQ010      0.028539
SMQ020      0.023724
LBXSAL      0.018263
RIDRETH1    0.017908
dtype: float32


In [ ]:
with open("biological_age_xgboost_clean.pkl", "wb") as f:
    pickle.dump({"model": model, "features": FEATURES}, f)
print("Saved: biological_age_xgboost_clean.pkl")

Saved: biological_age_xgboost_clean.pkl


: 